In [2]:
import probEDNR as edr
import numpy as np
import logging
from copy import deepcopy
import matplotlib.pyplot as plt
from matplotlib import pylab

# Solar STC(1kW/m2) nominal curve
spv_nomcurve=np.array([0]*6+[0.038, 0.188, 0.421, 0.676, 0.883, 0.988,
                     0.962, 0.812, 0.579, 0.324, 0.117, 0.012]+[0]*6)


## DEFINITION MG1 (kinda PCarreño) (mid-hi cost+mid-low cost)
# UNA CE municipal con agroindustria, relied on Diesel, has some biomass, recently got mid spv, no BESS. 5kWp resid curve
Diesel1=edr.Generator('Diesel',rate_copkwh=910,power_perunit_kw=[1750,1275,1000,1000,910,510])
Biomass1=edr.Generator('Biomass',480,[560]*4,UR=4*350,DR=4*350)
SPV1=edr.Generator('SPV',90,[250]*18,IBR=2,gen_curve_pu=spv_nomcurve) 
# Classic resid small city
demc1=np.array([0.79, 0.80, 0.77, 0.74, 0.73, 0.72, 0.69, 0.75, 0.78, 0.80, 0.85,
                 0.90, 0.91, 0.96, 1.00, 0.98, 0.96, 0.92, 0.94, 0.95, 0.97, 0.96, 0.93, 0.87])
MG1=edr.MG(5200,demc1,has_bess=False,Gens=(SPV1,Diesel1,Biomass1))
#18k-ish ppl
# pto carr has 20k ppl lol
print(MG1)

## DEFINITION MG2 (kinda Catatumbo) (mid-low cost+high cost)
# UNA CE REMOTA, mid-small PCH n mid Diesel, just got big wind with smallmid bess. 3.5kWp resid curve
Diesel2=edr.Generator('Diesel',820,[750,750,1250,1250],UR=2000,DR=2000)
PCH2=edr.Generator('PCH',380,[450]*2,UR=450*2,DR=450*2)
SPV2=edr.Generator('SPV',95,[250]*22,IBR=2,gen_curve_pu=spv_nomcurve*0.9) 
BESS_mid=edr.BESS(nmods=280) # Pylontech US5000
# Classic resid pueblo
demc2=np.array([0.57, 0.55, 0.54, 0.53, 0.55, 0.63, 0.71, 0.73, 0.75, 0.80, 0.82,
                 0.82, 0.83, 0.84, 0.83, 0.82, 0.79, 0.92, 0.96, 1.00, 0.96, 0.87, 0.67, 0.62])
MG2=edr.MG(3200,demc2,Gens=(SPV2,Diesel2,PCH2),has_bess=True,BESS=BESS_mid)
#13k-ish ppl
print(BESS_mid)
print(MG2)
print("\n")



## DEFINITION MG3
# UNA INDUSTRIAL, plenty coal, just bought big spv and big bess. 4kWp work curve, at night shit keeps running
Coal3=edr.Generator('Coal',780,[1000]*6,UR=500*6,DR=500*6)
SPV3=edr.Generator('SPV',85,[250]*14,IBR=2,gen_curve_pu=spv_nomcurve) 
WT_big=edr.Generator('WT',110,[100]*38,IBR=2,gen_curve_pu=[0.5]*12+[0.7]*12)
#Fortress Power eVault Max 18.5 kWh x nmods
lifecycle_optimism=1.25
BESS_big=edr.BESS(nmods=600,ch_eff=0.83,dc_eff=0.8,lifecycle=8000*lifecycle_optimism,capcost_copkwh=0.9e6,modulecap_kwh=0.85*18.5,modulepower_kw=9.2)
# Industrial but machines on at night
demc3=np.array([0.59, 0.60, 0.61, 0.60, 0.64, 0.69, 0.82, 0.96, 1.00, 0.99, 0.99,
                 0.97, 0.96, 0.95, 0.96, 0.92, 0.86, 0.83, 0.72, 0.65, 0.63, 0.64, 0.63, 0.62])
MG3=edr.MG(4500,demc3,True,Gens=(SPV3,WT_big,Coal3),BESS=BESS_big)
#15k-ish ppl equivalent, but yknow its industry
print(BESS_big)
print(MG3)

MG(peak_demand_kw=5200, demand_curve_pu=array([0.79, 0.8 , 0.77, 0.74, 0.73, 0.72, 0.69, 0.75, 0.78, 0.8 , 0.85,
       0.9 , 0.91, 0.96, 1.  , 0.98, 0.96, 0.92, 0.94, 0.95, 0.97, 0.96,
       0.93, 0.87]), demand_curve_kw=array([4108., 4160., 4004., 3848., 3796., 3744., 3588., 3900., 4056.,
       4160., 4420., 4680., 4732., 4992., 5200., 5096., 4992., 4784.,
       4888., 4940., 5044., 4992., 4836., 4524.]), has_bess=False, BESS=None, Gens=(Generator(type='SPV', rate_copkwh=90, power_perunit_kw=[250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250], power_kw=4500, gen_curve_pu=array([0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.038, 0.188, 0.421,
       0.676, 0.883, 0.988, 0.962, 0.812, 0.579, 0.324, 0.117, 0.012,
       0.   , 0.   , 0.   , 0.   , 0.   , 0.   ]), intermittent=True, dispatchable=False, IBR=2, UR=45000, DR=45000, T=24), Generator(type='Diesel', rate_copkwh=910, power_perunit_kw=[1750, 1275, 1000, 1000, 910, 510], power_kw=6445, gen_

In [8]:
mg=MG2
prefix="MG2"
hasbess=mg.has_bess

# SIMU PARAMS
global_init_params={"LastInstance":"Diesel"}
global_solver_params={
            'asymm_uses_MPO_not_multiplier':False,'RecMultiplier':1.2 # THIS IS THE DEFAULT
            }

# Sample Generator
temp=edr.detEDnR(mg,seed=42)

#### simples+bars
# Ntrainsamples=30
# Ntestsamples=2000
# Qset=85
# rwass=100
# TRAINSAMPLE=temp.generateSampleSet(Ntrainsamples)
# TESTSAMPLE=temp.generateSampleSet(Ntestsamples)
# TESTDAY=temp.generateDaySample()

#### simulate DROW barrido params
# TestQs=[50,75,90,100,110,125,150]
# Nrwass=13
# testrWs=list(set(np.logspace(1,4,num=Nrwass,base=10).astype(int)))
# testrWs.sort()
# Nrwass=len(testrWs)

#### runsimulations DROW Nexp
# Nexperiments=100
# TRAINSAMPLESET_DROW=[temp.generateSampleSet(Ntrainsamples) for _ in range(Nexperiments)]
# Ntestsamples=1000
# TESTSAMPLE_DROW=temp.generateSampleSet(Ntestsamples)

#### runsimulations DROW Nexp x barrido rwass
# TestQs_for_runsimus_drow=[50,75,100,125,150]
# TestRWs_for_runsimus_drow=testrWs

# Nexperiments=100
# TRAINSAMPLESET_DROW=[temp.generateSampleSet(Ntrainsamples) for _ in range(Nexperiments)]

#### runsimulations all meths x several params
TestQs_for_all=[50,75,100,125]
TestRWs_for_all=[100,1000,10000,100000]
Ntestsamples_szs=500
TESTSET_ALL=temp.generateSampleSet(Ntestsamples_szs)
Nexperiments=50
TRAINSAMPLESET_ALL=[]
# for _ in range(Nexperiments):
    # print(f"{_}")
    # TRAINSAMPLESET_ALL.append(temp.generateSampleSet(Ntrainsamples))

#### runsimulations all meths x several params x sample sizes
NsampSizes=11
MaxSampSize=500
# SampSizes=np.logspace(1.5,np.log10(MaxSampSize),num=NsampSizes,base=10).astype(int) #[ 31  41  54  72  95 125 165 218 287 379 499]
SampSizes= [72, 95, 125, 165, 218, 287, 379, 499]
print(SampSizes)
Nexperiments=50
TRAIN_SETS_PER_SIZE=[]
for sz in SampSizes:
    print(f"sz:{sz}")
    if sz>=72:
        TRAIN_SETS_PER_SIZE.append([temp.generateSampleSet(sz) for _ in range(Nexperiments)])


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2717763
Academic license 2717763 - for non-commercial use only - registered to sl___@unal.edu.co
[72, 95, 125, 165, 218, 287, 379, 499]
sz:72
sz:95
sz:125
sz:165
sz:218
sz:287
sz:379
sz:499


# Simulate just DROW barrido rwass

In [3]:
# Use these
# TestQs
# testrWs
# TRAINSAMPLE
# TESTSAMPLE

## RunSimulations just DROW

In [4]:
# use these
# TRAINSAMPLESET_DROW
# TESTSAMPLE_DROW
# Qset=85
# rwass=100






### now with diff params

In [5]:
# use these
# TRAINSAMPLESET_DROW
# TESTSAMPLE_DROW
# TestQs_for_runsimus_drow
# TestRWs_for_runsimus_drow


## RunSimulations For All Methods

In [9]:
###### PARAMS FOR ALL

TestQs_for_all=[50,75,100,125]
TestRWs_for_all=[100,1000,10000,100000]
nQs=len(TestQs_for_all)
Nrwass=len(TestRWs_for_all)
# use these
# TRAINSAMPLESET_ALL
# TESTSET_ALL

init_paramas={
              "logger_level":logging.CRITICAL
              }|global_init_params

# init methods
methods={"D":edr.detEDnR(mg,**init_paramas), "S": edr.SEDnR(mg,**init_paramas) ,
          "R": edr.REDnR(mg,**init_paramas) , "W": edr.DRWEDnR(mg,**init_paramas)}

# solver-related parameters
which_solver_to_use=["D"]+["S"]*nQs+["R"]*nQs+["W"]*nQs*Nrwass
solve_params=[{}] + [{"Q":q} for q in TestQs_for_all]*2 +\
    [{"Q":q,"rwass":rw} for q in TestQs_for_all for rw in TestRWs_for_all] 
for p in solve_params: p|=global_solver_params

MethodNames=["D"]+[f"S{q}" for q in TestQs_for_all]+[f"R{q}" for q in TestQs_for_all]+\
    [f"W{q}_{rw}" for q in TestQs_for_all for rw in TestRWs_for_all]



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2717763
Academic license 2717763 - for non-commercial use only - registered to sl___@unal.edu.co
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2717763
Academic license 2717763 - for non-commercial use only - registered to sl___@unal.edu.co
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2717763
Academic license 2717763 - for non-commercial use only - registered to sl___@unal.edu.co
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2717763
Academic license 2717763 - for non-commercial use only - registered to sl___@unal.edu.co


### One Sample Size

In [8]:
# Init results
metrics = ["jis", "joos", "prR", "rel"] + ["prB"]*hasbess
results_allmethods={n:{m:[] for m in metrics} for n in MethodNames}

# DET
solver=methods["D"]
name="D"
print(f"------ method: {name} ------")
all_my_solve_params=[solve_params[0]]
_xdec,Jis,Joos,rel,PrViol=solver.runSimulations(TRAINSAMPLESET_ALL,TESTSET_ALL,all_my_solve_params)
# results come as (Nparams,Nexp)
results_allmethods[name]["jis"]=np.array(Jis)[:,0] #now len Nexp
results_allmethods[name]["joos"]=np.array(Joos)[:,0] #now len Nexp
results_allmethods[name]['joos_jis']=results_allmethods[name]["joos"]-results_allmethods[name]["jis"]
results_allmethods[name]["rel"]=np.array(rel)[:,0] #now len Nexp
# guy comes as (Nparams,Nexp,1+hasbess)
prv_=np.array(PrViol)[:,0,:]
results_allmethods[name]["prR"]=prv_[:,0]
if hasbess:
    results_allmethods[name]["prB"]=prv_[:,1]

# S AND R
for i_,m in enumerate(["S", "R"]):
    print(f"------ method: {m} ------")
    solver=methods[m]
    all_my_solve_params=solve_params[1+(nQs*i_) : 1+(nQs*i_) + nQs]
    _xdec,Jis,Joos,rel,PrViol=solver.runSimulations(TRAINSAMPLESET_ALL,TESTSET_ALL,all_my_solve_params)
    # results come as (Nparams,Nexp)
    for i,name in enumerate(MethodNames[1+(nQs*i_) : 1+(nQs*i_) + nQs]):
        print(f"------ saving data for : {name} ------")
        results_allmethods[name]["jis"]=np.array(Jis)[:,i] #now len Nexp
        results_allmethods[name]["joos"]=np.array(Joos)[:,i] #now len Nexp
        results_allmethods[name]['joos_jis']=results_allmethods[name]["joos"]-results_allmethods[name]["jis"]
        results_allmethods[name]["rel"]=np.array(rel)[:,i] #now len Nexp
        # guy comes as (Nparams,Nexp,1+hasbess)
        prv_=np.array(PrViol)[:,i,:]
        results_allmethods[name]["prR"]=prv_[:,0]
        if hasbess:
            results_allmethods[name]["prB"]=prv_[:,1]

# DROW
solver=methods["W"]
print("------ method: W ------")
all_my_solve_params=solve_params[1+2*nQs:]
_xdec,Jis,Joos,rel,PrViol=solver.runSimulations(TRAINSAMPLESET_ALL,TESTSET_ALL,all_my_solve_params)
# results come as (Nparams,Nexp)
for i,name in enumerate(MethodNames[1+2*nQs:]):
    print(f"------ saving data for : {name} ------")
    results_allmethods[name]["jis"]=np.array(Jis)[:,i] #now len Nexp
    results_allmethods[name]["joos"]=np.array(Joos)[:,i] #now len Nexp
    results_allmethods[name]['joos_jis']=results_allmethods[name]["joos"]-results_allmethods[name]["jis"]
    results_allmethods[name]["rel"]=np.array(rel)[:,i] #now len Nexp
    # guy comes as (Nparams,Nexp,1+hasbess)
    prv_=np.array(PrViol)[:,i,:]
    results_allmethods[name]["prR"]=prv_[:,0]
    if hasbess:
        results_allmethods[name]["prB"]=prv_[:,1]
        
np.save(f"{prefix}_runsimus_allmeths_onesampsize_ASYMM1p2.npy", results_allmethods, allow_pickle=True)
del results_allmethods

[1577 - heuristic_reserve] Batt SOE bounds infeasible: 63.1% <= SOE <= 36.9%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=16.8%
[1577 - heuristic_reserve] Batt SOE bounds infeasible: 50.6% <= SOE <= 49.4%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=13.4%


------ method: D ------


[1577 - heuristic_reserve] Batt SOE bounds infeasible: 63.1% <= SOE <= 36.9%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=16.8%
[1577 - heuristic_reserve] Batt SOE bounds infeasible: 50.6% <= SOE <= 49.4%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=13.4%


------ method: S ------


[1577 - heuristic_reserve] Batt SOE bounds infeasible: 63.1% <= SOE <= 36.9%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=16.8%
[1577 - heuristic_reserve] Batt SOE bounds infeasible: 50.6% <= SOE <= 49.4%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=13.4%


------ saving data for : S50 ------
------ saving data for : S75 ------
------ saving data for : S100 ------
------ saving data for : S125 ------
------ method: R ------


[1577 - heuristic_reserve] Batt SOE bounds infeasible: 63.1% <= SOE <= 36.9%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=16.8%
[1577 - heuristic_reserve] Batt SOE bounds infeasible: 50.6% <= SOE <= 49.4%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=13.4%


------ saving data for : R50 ------
------ saving data for : R75 ------
------ saving data for : R100 ------
------ saving data for : R125 ------
------ method: W ------
------ saving data for : W50_100 ------
------ saving data for : W50_1000 ------
------ saving data for : W50_10000 ------
------ saving data for : W50_100000 ------
------ saving data for : W75_100 ------
------ saving data for : W75_1000 ------
------ saving data for : W75_10000 ------
------ saving data for : W75_100000 ------
------ saving data for : W100_100 ------
------ saving data for : W100_1000 ------
------ saving data for : W100_10000 ------
------ saving data for : W100_100000 ------
------ saving data for : W125_100 ------
------ saving data for : W125_1000 ------
------ saving data for : W125_10000 ------
------ saving data for : W125_100000 ------


### All Sample Sizes

In [10]:
metrics = ["jis_avg", "jis_q25","jis_q75",
           "joos_avg", "joos_q25","joos_q75",
        "prR_avg", "prR_q25","prR_q75",
        "rel"] +\
        ["joos_jis_avg", "joos_jis_q25","joos_jis_q75",] +\
              ["prB_avg","prB_q25", "prB_q75"]*hasbess
getters=["avgJis" , "q25Jis" , "q75Jis" , 
         "avgJoos","q25Joos","q75Joos",
         "avgPrViol","q25Prviol","q75Prviol",
         "avgRel"]

results_big={n:{m:[] for m in metrics} for n in MethodNames}

# ITERATE
for isz,sz in enumerate(SampSizes):
    print(f"====== Nsamples: {sz} ======")
    for i,params in enumerate(solve_params):
        solver=methods[which_solver_to_use[i]]
        name=MethodNames[i]
        print(f"------ method: {name} ------")
        TrainSets = TRAIN_SETS_PER_SIZE[isz]
        _xdec,Jis,Joos,rel,PrViol=solver.runSimulations(TrainSets,TESTSET_ALL,[params])

        for i,m in enumerate(metrics[:10]):
            # print(f"name:{name} ==== m: {m}")
            results_big[name][m].append(solver.__getattribute__(getters[i])[0])

        Joos_Jis=np.array(Joos)-np.array(Jis)
        # print(f"Joos_Jis: {Joos_Jis}")
        results_big[name]["joos_jis_avg"].append(np.mean(Joos_Jis))
        results_big[name]["joos_jis_q25"].append(np.quantile(Joos_Jis,0.25))
        results_big[name]["joos_jis_q75"].append(np.quantile(Joos_Jis,0.75))

        # print(f"res before : {results_big[name]}")

        if hasbess:
            results_big[name]["prB_avg"].append(results_big[name]["prR_avg"][isz][1])
            results_big[name]["prB_q25"].append(results_big[name]["prR_q25"][isz][1])
            results_big[name]["prB_q75"].append(results_big[name]["prR_q75"][isz][1])
        results_big[name]["prR_avg"][isz]=results_big[name]["prR_avg"][isz][0]
        results_big[name]["prR_q25"][isz]=results_big[name]["prR_q25"][isz][0]
        results_big[name]["prR_q75"][isz]=results_big[name]["prR_q75"][isz][0]

        # print(f"res : {results_big[name]}")
results_big_numpyify={m:{k:np.array(vec) for k,vec in data.items()} for m,data in results_big.items()}
np.save(f"{prefix}_runsimus_allmeths_allsampsizes_ASYMM1p2_fromN72.npy",results_big_numpyify, allow_pickle=True)
del results_big_numpyify

====== Nsamples: 72 ======
------ method: D ------


[1577 - heuristic_reserve] Batt SOE bounds infeasible: 63.1% <= SOE <= 36.9%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=16.8%
[1577 - heuristic_reserve] Batt SOE bounds infeasible: 50.6% <= SOE <= 49.4%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=13.4%
[1577 - heuristic_reserve] Batt SOE bounds infeasible: 63.1% <= SOE <= 36.9%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=16.8%
[1577 - heuristic_reserve] Batt SOE bounds infeasible: 50.6% <= SOE <= 49.4%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=13.4%


------ method: S50 ------
------ method: S75 ------
------ method: S100 ------
------ method: S125 ------


[1577 - heuristic_reserve] Batt SOE bounds infeasible: 63.1% <= SOE <= 36.9%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=16.8%
[1577 - heuristic_reserve] Batt SOE bounds infeasible: 50.6% <= SOE <= 49.4%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=13.4%


------ method: R50 ------
------ method: R75 ------
------ method: R100 ------
------ method: R125 ------


...

------ method: W50_1000 ------
------ method: W50_10000 ------
------ method: W50_100000 ------
------ method: W75_100 ------
------ method: W75_1000 ------
------ method: W75_10000 ------
------ method: W75_100000 ------
------ method: W100_100 ------
------ method: W100_1000 ------
------ method: W100_10000 ------
------ method: W100_100000 ------
------ method: W125_100 ------
------ method: W125_1000 ------
------ method: W125_10000 ------
------ method: W125_100000 ------
====== Nsamples: 499 ======
------ method: D ------
------ method: S50 ------
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2717763
Academic license 2717763 - for non-commercial use only - registered to sl___@unal.edu.co


[1577 - heuristic_reserve] Batt SOE bounds infeasible: 63.1% <= SOE <= 36.9%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=16.8%
[1577 - heuristic_reserve] Batt SOE bounds infeasible: 50.6% <= SOE <= 49.4%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=13.4%


------ method: S75 ------
------ method: S100 ------
------ method: S125 ------
------ method: R50 ------
------ method: R75 ------
------ method: R100 ------
------ method: R125 ------
------ method: W50_100 ------
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2717763
Academic license 2717763 - for non-commercial use only - registered to sl___@unal.edu.co


[1577 - heuristic_reserve] Batt SOE bounds infeasible: 63.1% <= SOE <= 36.9%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=16.8%
[1577 - heuristic_reserve] Batt SOE bounds infeasible: 50.6% <= SOE <= 49.4%.
[1589 - heuristic_reserve] Retrying with lower fpart_bess=13.4%


------ method: W50_1000 ------
------ method: W50_10000 ------
------ method: W50_100000 ------
------ method: W75_100 ------
------ method: W75_1000 ------
------ method: W75_10000 ------
------ method: W75_100000 ------
------ method: W100_100 ------
------ method: W100_1000 ------
------ method: W100_10000 ------
------ method: W100_100000 ------
------ method: W125_100 ------
------ method: W125_1000 ------
------ method: W125_10000 ------
------ method: W125_100000 ------


In [15]:
results_big

{'jis_avg': [34913829.22882698,
  34926674.79706699,
  34919214.145746976,
  34938093.97214698],
 'jis_q25': [34570962.65140697,
  34631763.34790698,
  34734793.34790698,
  34741942.09790698],
 'jis_q75': [35223540.84790698,
  35183673.34790698,
  35172295.84790698,
  35077667.09790698],
 'joos_avg': [116345765.03115025,
  116147540.31509277,
  116113179.94991066,
  116116069.83697626],
 'joos_q25': [115013682.30050328,
  115046678.71655357,
  115350926.5631703,
  115259312.99249563],
 'joos_q75': [118152677.78795196,
  117278778.20577249,
  116951530.48924735,
  116969491.57122567],
 'prR_avg': [0.23852683333333338,
  0.23852672222222218,
  0.23852577777777773,
  0.2385143333333333],
 'prR_q25': [0.23850069444444444,
  0.23850347222222223,
  0.23850277777777776,
  0.23850277777777776],
 'prR_q75': [0.2385333333333333,
  0.23853888888888888,
  0.23853819444444443,
  0.23852430555555554],
 'rel': [0.0, 0.0, 0.0, 0.0],
 'joos_jis_avg': [81431935.80232328,
  81220865.5180258,
  81193965.8

In [16]:
results_big_numpyify={m:{k:np.array(vec) for k,vec in data.items()} for m,data in results_big.items()}
# results_big_numpyify["D"]["jis_avg"]
np.save(f"{prefix}_runsimus_allmeths_allsampsizes_ASYMM1p2_almost_done_N72.npy",results_big_numpyify, allow_pickle=True)

In [8]:
x=np.load("MG3_runsimus_allmeths_allsampsizes_ASYMM1p2.npy",allow_pickle=True)[()]
x["W100_100000"]

{'jis_avg': array([34165112.31324878, 33878020.94685934, 33999155.16838806,
        33977337.82443316, 33946903.39870507, 33929100.824691  ,
        33935055.79291017, 33896922.91164943, 33898518.53555755,
        33727528.30896225, 33794000.1838303 ]),
 'jis_q25': array([33229719.85613934, 33109404.34706064, 32971609.42057755,
        33495279.91677194, 33553875.78361013, 33490482.86369998,
        33655359.6518978 , 33663162.94844735, 33662202.88656111,
        33398134.43553131, 33585366.09681138]),
 'jis_q75': array([35111243.34791425, 34780717.24924093, 34863760.948011  ,
        34545876.96479492, 34501419.1334523 , 34351970.64965633,
        34321237.88694046, 34170745.74069277, 34155151.88326983,
        34097797.9099149 , 33995336.27721532]),
 'joos_avg': array([32024952.16749624, 31520213.8113429 , 31152964.53974208,
        31142307.37926534, 31421867.05260462, 31219512.36366438,
        30616681.69910049, 30714469.29822014, 30897792.07760568,
        30465759.43328476, 3030

## BIG BOY LOOP - ONE SAMP SIZE AND ALL SAMP SIZES - FOR THE 3 MGs

In [1]:
## IMPORT LIBRARIES CREATE MGs
import probEDNR as edr
import numpy as np
import logging
from copy import deepcopy
import matplotlib.pyplot as plt
from matplotlib import pylab

# Solar STC(1kW/m2) nominal curve
spv_nomcurve=np.array([0]*6+[0.038, 0.188, 0.421, 0.676, 0.883, 0.988,
                     0.962, 0.812, 0.579, 0.324, 0.117, 0.012]+[0]*6)


## DEFINITION MG1 (kinda PCarreño) (mid-hi cost+mid-low cost)
# UNA CE municipal con agroindustria, relied on Diesel, has some biomass, recently got mid spv, no BESS. 5kWp resid curve
Diesel1=edr.Generator('Diesel',rate_copkwh=910,power_perunit_kw=[1750,1275,1000,1000,910,510])
Biomass1=edr.Generator('Biomass',480,[560]*4,UR=4*350,DR=4*350)
SPV1=edr.Generator('SPV',90,[250]*18,IBR=2,gen_curve_pu=spv_nomcurve) 
# Classic resid small city
demc1=np.array([0.79, 0.80, 0.77, 0.74, 0.73, 0.72, 0.69, 0.75, 0.78, 0.80, 0.85,
                 0.90, 0.91, 0.96, 1.00, 0.98, 0.96, 0.92, 0.94, 0.95, 0.97, 0.96, 0.93, 0.87])
MG1=edr.MG(5200,demc1,has_bess=False,Gens=(SPV1,Diesel1,Biomass1))
#18k-ish ppl
# pto carr has 20k ppl lol
# print(MG1)

## DEFINITION MG2 (kinda Catatumbo) (mid-low cost+high cost)
# UNA CE REMOTA, mid-small PCH n mid Diesel, just got big wind with smallmid bess. 3.5kWp resid curve
Diesel2=edr.Generator('Diesel',820,[750,750,1250,1250],UR=2000,DR=2000)
PCH2=edr.Generator('PCH',380,[450]*2,UR=450*2,DR=450*2)
SPV2=edr.Generator('SPV',95,[250]*22,IBR=2,gen_curve_pu=spv_nomcurve*0.9) 
BESS_mid=edr.BESS(nmods=280) # Pylontech US5000
# Classic resid pueblo
demc2=np.array([0.57, 0.55, 0.54, 0.53, 0.55, 0.63, 0.71, 0.73, 0.75, 0.80, 0.82,
                 0.82, 0.83, 0.84, 0.83, 0.82, 0.79, 0.92, 0.96, 1.00, 0.96, 0.87, 0.67, 0.62])
MG2=edr.MG(3200,demc2,Gens=(SPV2,Diesel2,PCH2),has_bess=True,BESS=BESS_mid)
#13k-ish ppl
# print(BESS_mid)
# print(MG2)
# print("\n")



## DEFINITION MG3
# UNA INDUSTRIAL, plenty coal, just bought big spv and big bess. 4kWp work curve, at night shit keeps running
Coal3=edr.Generator('Coal',780,[1000]*6,UR=500*6,DR=500*6)
SPV3=edr.Generator('SPV',85,[250]*14,IBR=2,gen_curve_pu=spv_nomcurve) 
WT_big=edr.Generator('WT',110,[100]*38,IBR=2,gen_curve_pu=[0.5]*12+[0.7]*12)
#Fortress Power eVault Max 18.5 kWh x nmods
lifecycle_optimism=1.25
BESS_big=edr.BESS(nmods=600,ch_eff=0.83,dc_eff=0.8,lifecycle=8000*lifecycle_optimism,capcost_copkwh=0.9e6,modulecap_kwh=0.85*18.5,modulepower_kw=9.2)
# Industrial but machines on at night
demc3=np.array([0.59, 0.60, 0.61, 0.60, 0.64, 0.69, 0.82, 0.96, 1.00, 0.99, 0.99,
                 0.97, 0.96, 0.95, 0.96, 0.92, 0.86, 0.83, 0.72, 0.65, 0.63, 0.64, 0.63, 0.62])
MG3=edr.MG(4500,demc3,True,Gens=(SPV3,WT_big,Coal3),BESS=BESS_big)
#15k-ish ppl equivalent, but yknow its industry
# print(BESS_big)
# print(MG3)

In [ ]:
li=["Diesel","Diesel","Coal"]
for i_mg, mg in enumerate([MG1,MG2,MG3]):
    ### NAMESTUFF
    prefix=f"MG{i_mg+1}"
    print(f"================{prefix}==================")
    hasbess=mg.has_bess
    global_init_params={"LastInstance":li[i_mg], "logger_level":logging.CRITICAL, "logger_scope":2,'grb_verbose':False}
    global_solver_params={
                'asymm_uses_MPO_not_multiplier':False,'RecMultiplier':1.2 # THIS IS THE DEFAULT, JUST IN CASE
                }
    
    ### TEST QS AND RWS
    TestQs_for_all=[50,75,90,95]
    # TestQs_for_all=[95,100] #### TOY
    TestRWs_for_all=[100,500,1000,5000,10000,50000,100000]
    # TestRWs_for_all=[5000,100000] #### TOY
    nQs=len(TestQs_for_all)
    Nrwass=len(TestRWs_for_all)
    
    ### MAKE SAMPLES
    temp=edr.detEDnR(mg,seed=50,grb_verbose=False)
    Ntestsamples_szs=500
    # Ntestsamples_szs=10 #### TOY
    TESTSET_ALL=temp.generateSampleSet(Ntestsamples_szs)
    Nexperiments=50
    # Nexperiments=10 #### TOY
    
    ## one train size
    Ntrainsamples=30
    # Ntrainsamples=10 #### TOY
    TRAINSAMPLESET_ALL=[]
    for _ in range(Nexperiments):
        TRAINSAMPLESET_ALL.append(temp.generateSampleSet(Ntrainsamples))
    print(f"Created set of sample sets, sz: {Ntrainsamples}")
    ## all train sizes
    SampSizes=[ 31 , 41 , 54 , 72 , 95 ,125, 165, 218 ,287, 379 ,499]
    # SampSizes=[3,100] #### TOY
    TRAIN_SETS_PER_SIZE=[]
    for sz in SampSizes:
        print(f"creating samp set of sz:{sz}")
        TRAIN_SETS_PER_SIZE.append([temp.generateSampleSet(sz) for _ in range(Nexperiments)])

    ### INITIALIZE SHIT
    # init methods
    methods={"D":edr.detEDnR(mg,**global_init_params), "S": edr.SEDnR(mg,**global_init_params) ,
              "R": edr.REDnR(mg,**global_init_params) , "W": edr.DRWEDnR(mg,**global_init_params)}
    
    # solver-related parameters
    which_solver_to_use=["D"]+["S"]*nQs+["R"]*nQs+["W"]*nQs*Nrwass
    solve_params=[{}] + [{"Q":q} for q in TestQs_for_all]*2 +\
        [{"Q":q,"rwass":rw} for q in TestQs_for_all for rw in TestRWs_for_all] 
    for p in solve_params: p|=global_solver_params
    
    MethodNames=["D"]+[f"S{q}" for q in TestQs_for_all]+[f"R{q}" for q in TestQs_for_all]+\
        [f"W{q}_{rw}" for q in TestQs_for_all for rw in TestRWs_for_all]

    
    print(f"================{prefix} ONE SAMPLE SIZE ==================")
    
    ### ONE SAMPLE SIZE    
    # Init results
    metrics = ["jis", "joos", "prR", "rel"] + ["prB"]*hasbess
    results_allmethods={n:{m:[] for m in metrics} for n in MethodNames}
    
    # DET
    solver=methods["D"]
    name="D"
    print(f"------------ method: {name} ------------")
    all_my_solve_params=[solve_params[0]]
    _xdec,Jis,Joos,rel,PrViol=solver.runSimulations(TRAINSAMPLESET_ALL,TESTSET_ALL,all_my_solve_params)
    # results come as (Nparams,Nexp)
    results_allmethods[name]["jis"]=np.array(Jis)[:,0] #now len Nexp
    results_allmethods[name]["joos"]=np.array(Joos)[:,0] #now len Nexp
    results_allmethods[name]['joos_jis']=results_allmethods[name]["joos"]-results_allmethods[name]["jis"]
    results_allmethods[name]["rel"]=np.array(rel)[:,0] #now len Nexp
    # guy comes as (Nparams,Nexp,1+hasbess)
    prv_=np.array(PrViol)[:,0,:]
    results_allmethods[name]["prR"]=prv_[:,0]
    if hasbess:
        results_allmethods[name]["prB"]=prv_[:,1]
    
    # S AND R
    for i_,m in enumerate(["S", "R"]):
        print(f"------------ method: {m} ------------")
        solver=methods[m]
        all_my_solve_params=solve_params[1+(nQs*i_) : 1+(nQs*i_) + nQs]
        _xdec,Jis,Joos,rel,PrViol=solver.runSimulations(TRAINSAMPLESET_ALL,TESTSET_ALL,all_my_solve_params)
        # results come as (Nparams,Nexp)
        for i,name in enumerate(MethodNames[1+(nQs*i_) : 1+(nQs*i_) + nQs]):
            print(f"------ saving data for : {name} ------")
            results_allmethods[name]["jis"]=np.array(Jis)[:,i] #now len Nexp
            results_allmethods[name]["joos"]=np.array(Joos)[:,i] #now len Nexp
            results_allmethods[name]['joos_jis']=results_allmethods[name]["joos"]-results_allmethods[name]["jis"]
            results_allmethods[name]["rel"]=np.array(rel)[:,i] #now len Nexp
            # guy comes as (Nparams,Nexp,1+hasbess)
            prv_=np.array(PrViol)[:,i,:]
            results_allmethods[name]["prR"]=prv_[:,0]
            if hasbess:
                results_allmethods[name]["prB"]=prv_[:,1]
    
    # DROW
    solver=methods["W"]
    print("------------ method: W ------------")
    all_my_solve_params=solve_params[1+2*nQs:]
    _xdec,Jis,Joos,rel,PrViol=solver.runSimulations(TRAINSAMPLESET_ALL,TESTSET_ALL,all_my_solve_params)
    # results come as (Nparams,Nexp)
    for i,name in enumerate(MethodNames[1+2*nQs:]):
        print(f"------ saving data for : {name} ------")
        results_allmethods[name]["jis"]=np.array(Jis)[:,i] #now len Nexp
        results_allmethods[name]["joos"]=np.array(Joos)[:,i] #now len Nexp
        results_allmethods[name]['joos_jis']=results_allmethods[name]["joos"]-results_allmethods[name]["jis"]
        results_allmethods[name]["rel"]=np.array(rel)[:,i] #now len Nexp
        # guy comes as (Nparams,Nexp,1+hasbess)
        prv_=np.array(PrViol)[:,i,:]
        results_allmethods[name]["prR"]=prv_[:,0]
        if hasbess:
            results_allmethods[name]["prB"]=prv_[:,1]
            
    np.save(f"{prefix}_runsimus_allmeths_onesampsize_ASYMM1p2.npy", results_allmethods, allow_pickle=True)
    del results_allmethods


    #### ALL SAMPLE SIZES
    
    print(f"================{prefix} ALL SAMPLE SIZES ==================")

    metrics = ["jis_avg", "jis_q25","jis_q75",
               "joos_avg", "joos_q25","joos_q75",
            "prR_avg", "prR_q25","prR_q75",
            "rel"] +\
            ["joos_jis_avg", "joos_jis_q25","joos_jis_q75",] +\
                  ["prB_avg","prB_q25", "prB_q75"]*hasbess
    getters=["avgJis" , "q25Jis" , "q75Jis" , 
             "avgJoos","q25Joos","q75Joos",
             "avgPrViol","q25Prviol","q75Prviol",
             "avgRel"]
    
    results_big={n:{m:[] for m in metrics} for n in MethodNames}
    
    # ITERATE
    for isz,sz in enumerate(SampSizes):
        print(f"============ Nsamples: {sz} ============")
        for i,params in enumerate(solve_params):
            solver=methods[which_solver_to_use[i]]
            name=MethodNames[i]
            print(f"------------ method: {name} ------------")
            TrainSets = TRAIN_SETS_PER_SIZE[isz]
            _xdec,Jis,Joos,rel,PrViol=solver.runSimulations(TrainSets,TESTSET_ALL,[params])
    
            for i,m in enumerate(metrics[:10]):
                # print(f"name:{name} ==== m: {m}")
                results_big[name][m].append(solver.__getattribute__(getters[i])[0])
    
            Joos_Jis=np.array(Joos)-np.array(Jis)
            # print(f"Joos_Jis: {Joos_Jis}")
            results_big[name]["joos_jis_avg"].append(np.mean(Joos_Jis))
            results_big[name]["joos_jis_q25"].append(np.quantile(Joos_Jis,0.25))
            results_big[name]["joos_jis_q75"].append(np.quantile(Joos_Jis,0.75))
    
            # print(f"res before : {results_big[name]}")
    
            if hasbess:
                results_big[name]["prB_avg"].append(results_big[name]["prR_avg"][isz][1])
                results_big[name]["prB_q25"].append(results_big[name]["prR_q25"][isz][1])
                results_big[name]["prB_q75"].append(results_big[name]["prR_q75"][isz][1])
            results_big[name]["prR_avg"][isz]=results_big[name]["prR_avg"][isz][0]
            results_big[name]["prR_q25"][isz]=results_big[name]["prR_q25"][isz][0]
            results_big[name]["prR_q75"][isz]=results_big[name]["prR_q75"][isz][0]
    
            # print(f"res : {results_big[name]}")
    results_big_numpyify={m:{k:np.array(vec) for k,vec in data.items()} for m,data in results_big.items()}
    np.save(f"{prefix}_runsimus_allmeths_allsampsizes_ASYMM1p2.npy",results_big_numpyify, allow_pickle=True)
    del results_big_numpyify

print("\n===========================FINISHED===========================")


================MG1==================
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2717763
Academic license 2717763 - for non-commercial use only - registered to sl___@unal.edu.co
Created set of sample sets, sz: 30
creating samp set of sz:31
creating samp set of sz:41
creating samp set of sz:54
creating samp set of sz:72
creating samp set of sz:95
creating samp set of sz:125
creating samp set of sz:165
creating samp set of sz:218
creating samp set of sz:287
creating samp set of sz:379
creating samp set of sz:499
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2717763
Academic license 2717763 - for non-commercial use only - registered to sl___@unal.edu.co
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2717763
Academic license 2717763 - for non-commercial use only - registered to sl___@unal.edu.co
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2

###########################################

In [18]:
# np.load("MG1_runsimus_allmeths_onesampsize_ASYMM1p2.npy",allow_pickle=True)[()]
l=np.load("MG1_runsimus_allmeths_allsampsizes_ASYMM1p2.npy",allow_pickle=True)[()]
l["W50_100"]
l["W50_5000"]

{'jis_avg': array([66716662.95706072, 58705216.44716197]),
 'jis_q25': array([60495428.6840833 , 55451099.78972443]),
 'jis_q75': array([73092741.7652725 , 62630264.12237947]),
 'joos_avg': array([62013324.31131484, 60268958.04487462]),
 'joos_q25': array([60604290.09015372, 59611884.31961776]),
 'joos_q75': array([63111281.42217076, 61037145.97945998]),
 'prR_avg': array([0.64055556, 0.66427778]),
 'prR_q25': array([0.59861111, 0.63805556]),
 'prR_q75': array([0.72138889, 0.66972222]),
 'rel': array([0.6, 0.6]),
 'joos_jis_avg': array([-4703338.64574587,  1563741.59771265]),
 'joos_jis_q25': array([-12890375.00873519,  -3018379.8027617 ]),
 'joos_jis_q75': array([ 108861.40607042, 4929226.03424807])}

## ONE SAMP SIZE ALL REC MODES

In [46]:
mg=MG2 # ONLY DO FOR 1 AND 2
i_mg=1
prefix=f"MG{i_mg+1}"
print(f"================{prefix}==================")
hasbess=mg.has_bess
TestQs_for_recmodes=[50,75,90,95]
TestRWs_for_recmodes=[500,5000,50000] # not all
REC_MODE_names=["sym","asym_m12","asym_m15","asym_m20","asym_mpo"]

================MG2==================


In [56]:
## ONE SAMP SIZE ALL REC MODES
li=["Diesel","Diesel","Coal"]
### NAMESTUFF
prefix=f"MG{i_mg+1}"
print(f"================{prefix}==================")
hasbess=mg.has_bess

### TEST QS AND RWS
TestQs_for_recmodes=[50,75,90,95]
#TestQs_for_recmodes=[95,100] #### TOY
TestRWs_for_recmodes=[500,5000,50000] # not all
#TestRWs_for_recmodes=[5000,100000] #### TOY
nQs=len(TestQs_for_recmodes)
Nrwass=len(TestRWs_for_recmodes)

### MAKE SAMPLES
temp=edr.detEDnR(mg,seed=42,grb_verbose=False)
Ntestsamples_szs=200 # <500
#Ntestsamples_szs=10 #### TOY
TESTSET_ALL=temp.generateSampleSet(Ntestsamples_szs)
Nexperiments=30 # <50
#Nexperiments=10 #### TOY

## one train size
Ntrainsamples=30
#Ntrainsamples=10 #### TOY
TRAINSAMPLESET_ALL=[]
for _ in range(Nexperiments):
    TRAINSAMPLESET_ALL.append(temp.generateSampleSet(Ntrainsamples))
print(f"Created set of sample sets, sz: {Ntrainsamples}")


### INITIALIZE SHIT

# solver-related parameters
which_solver_to_use=["D"]+["S"]*nQs+["R"]*nQs+["W"]*nQs*Nrwass
MethodNames=["D"]+[f"S{q}" for q in TestQs_for_recmodes]+[f"R{q}" for q in TestQs_for_recmodes]+\
    [f"W{q}_{rw}" for q in TestQs_for_recmodes for rw in TestRWs_for_recmodes]

REC_MODE_names=["sym","asym_m12","asym_m15","asym_m20","asym_mpo"]
REC_MODES=[{'asymm_Rec':False},
           {'asymm_uses_MPO_not_multiplier':False,'RecMultiplier':1.2},
           {'asymm_uses_MPO_not_multiplier':False,'RecMultiplier':1.5},
           {'asymm_uses_MPO_not_multiplier':False,'RecMultiplier':2},
           {'asymm_uses_MPO_not_multiplier':True}]

for i_save,global_solver_params in enumerate(REC_MODES):

    recname=REC_MODE_names[i_save]
    
    # init methods
    global_init_params={"LastInstance":li[i_mg], "logger_level":logging.CRITICAL, "logger_scope":1,'grb_verbose':False}
    if recname=="asym_mpo": global_init_params["needMPO"]=True
    methods={"D":edr.detEDnR(mg,**global_init_params), "S": edr.SEDnR(mg,**global_init_params) ,
              "R": edr.REDnR(mg,**global_init_params) , "W": edr.DRWEDnR(mg,**global_init_params)}

    print(f"================ with REC {recname} ==================")
    
    solve_params=[{}] + [{"Q":q} for q in TestQs_for_recmodes]*2 +\
        [{"Q":q,"rwass":rw} for q in TestQs_for_recmodes for rw in TestRWs_for_recmodes] 
    for p in solve_params: p|=global_solver_params
    
    
    ### ONE SAMPLE SIZE    
    # Init results
    metrics = ["jis", "joos", "prR", "rel"] + ["prB"]*hasbess
    results_allmethods={n:{m:[] for m in metrics} for n in MethodNames}
    
    # DET
    solver=methods["D"]
    name="D"
    print(f"------------ method: {name} ------------")
    all_my_solve_params=[solve_params[0]]
    _xdec,Jis,Joos,rel,PrViol=solver.runSimulations(TRAINSAMPLESET_ALL,TESTSET_ALL,all_my_solve_params)
    # results come as (Nparams,Nexp)
    results_allmethods[name]["jis"]=np.array(Jis)[:,0] #now len Nexp
    results_allmethods[name]["joos"]=np.array(Joos)[:,0] #now len Nexp
    results_allmethods[name]['joos_jis']=results_allmethods[name]["joos"]-results_allmethods[name]["jis"]
    results_allmethods[name]["rel"]=np.array(rel)[:,0] #now len Nexp
    # guy comes as (Nparams,Nexp,1+hasbess)
    prv_=np.array(PrViol)[:,0,:]
    results_allmethods[name]["prR"]=prv_[:,0]
    if hasbess:
        results_allmethods[name]["prB"]=prv_[:,1]
    
    # S AND R
    for i_,m in enumerate(["S", "R"]):
        print(f"------------ method: {m} ------------")
        solver=methods[m]
        all_my_solve_params=solve_params[1+(nQs*i_) : 1+(nQs*i_) + nQs]
        _xdec,Jis,Joos,rel,PrViol=solver.runSimulations(TRAINSAMPLESET_ALL,TESTSET_ALL,all_my_solve_params)
        # results come as (Nparams,Nexp)
        for i,name in enumerate(MethodNames[1+(nQs*i_) : 1+(nQs*i_) + nQs]):
            print(f"------ saving data for : {name} ------")
            results_allmethods[name]["jis"]=np.array(Jis)[:,i] #now len Nexp
            results_allmethods[name]["joos"]=np.array(Joos)[:,i] #now len Nexp
            results_allmethods[name]['joos_jis']=results_allmethods[name]["joos"]-results_allmethods[name]["jis"]
            results_allmethods[name]["rel"]=np.array(rel)[:,i] #now len Nexp
            # guy comes as (Nparams,Nexp,1+hasbess)
            prv_=np.array(PrViol)[:,i,:]
            results_allmethods[name]["prR"]=prv_[:,0]
            if hasbess:
                results_allmethods[name]["prB"]=prv_[:,1]
    
    # DROW
    solver=methods["W"]
    print("------------ method: W ------------")
    all_my_solve_params=solve_params[1+2*nQs:]
    _xdec,Jis,Joos,rel,PrViol=solver.runSimulations(TRAINSAMPLESET_ALL,TESTSET_ALL,all_my_solve_params)
    # results come as (Nparams,Nexp)
    for i,name in enumerate(MethodNames[1+2*nQs:]):
        print(f"------ saving data for : {name} ------")
        results_allmethods[name]["jis"]=np.array(Jis)[:,i] #now len Nexp
        results_allmethods[name]["joos"]=np.array(Joos)[:,i] #now len Nexp
        results_allmethods[name]['joos_jis']=results_allmethods[name]["joos"]-results_allmethods[name]["jis"]
        results_allmethods[name]["rel"]=np.array(rel)[:,i] #now len Nexp
        # guy comes as (Nparams,Nexp,1+hasbess)
        prv_=np.array(PrViol)[:,i,:]
        results_allmethods[name]["prR"]=prv_[:,0]
        if hasbess:
            results_allmethods[name]["prB"]=prv_[:,1]
            
    np.save(f"{prefix}_runsimus_allmeths_onesampsize_rec_{recname}.npy", results_allmethods, allow_pickle=True)
    del results_allmethods

print("\n===========================FINISHED===========================")

================MG2==================
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2717763
Academic license 2717763 - for non-commercial use only - registered to sl___@unal.edu.co
Set parameter LogToConsole to value 0
Created set of sample sets, sz: 30
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2717763
Academic license 2717763 - for non-commercial use only - registered to sl___@unal.edu.co
Set parameter LogToConsole to value 0
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2717763
Academic license 2717763 - for non-commercial use only - registered to sl___@unal.edu.co
Set parameter LogToConsole to value 0
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2717763
Academic license 2717763 - for non-commercial use only - registered to sl___@unal.edu.co
Set parameter LogToConsole to value 0
Set parameter WLSAccessID
Set parameter WLSSecret
Set p